In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import glob
import os
from clinic_Functions.limpeza_dados import pipeline_completo 
from clinic_Functions.filtros import separar_todos_predios
from clinic_Functions.exportacao import exportar_dados_separados

In [ ]:
import glob
import os
import pandas as pd
import gc 

pasta_origem = "/home/ls/TCC_2026/Dados greefra/mes_1/dados de energia"
pasta_destino = "/home/ls/TCC_2026/Dados greefra/mes_1/dados_limpos"

if not os.path.exists(pasta_destino):
    os.makedirs(pasta_destino)

arquivos_csv = glob.glob(os.path.join(pasta_origem, "*.csv"))
print(f"Encontrados {len(arquivos_csv)} ficheiros. A iniciar processamento em LOTES DE LINHAS (Chunks)...\n")

# Define quantas linhas ler de cada vez (10.000 é um valor muito seguro para pouca RAM)
tamanho_lote = 10000

for arquivo in arquivos_csv:
    nome_arquivo = os.path.basename(arquivo)
    print(f"\n--- A processar o ficheiro: {nome_arquivo} ---")
    
    # O chunksize faz o read_csv retornar um iterador em vez do ficheiro todo
    for i, df_chunk in enumerate(pd.read_csv(arquivo, chunksize=tamanho_lote)):
        print(f"  -> A processar o lote {i + 1} (linhas {i * tamanho_lote} a {(i + 1) * tamanho_lote})...")
        
        # 1. Executa a limpeza e expansão APENAS nestas 10.000 linhas
        df_limpo = pipeline_completo(df_chunk)
        
        # 2. Separa por prédio as linhas deste pequeno lote
        tabelas_predios = separar_todos_predios(df_limpo, coluna='item_name')
        
        # 3. Guarda diretamente no ficheiro final do prédio (modo 'append')
        for nome_predio, df_predio in tabelas_predios.items():
            nome_arquivo_predio = str(nome_predio).replace(" ", "_") + ".csv"
            caminho_completo = os.path.join(pasta_destino, nome_arquivo_predio)
            
            if os.path.exists(caminho_completo):
                df_predio.to_csv(caminho_completo, mode='a', index=False, header=False)
            else:
                df_predio.to_csv(caminho_completo, mode='w', index=False, header=True)
                
        # 4. Limpa a memória instantaneamente após processar o lote
        del df_chunk
        del df_limpo
        del tabelas_predios
        gc.collect()

print("\nProcesso concluído com sucesso! Os dados foram processados sem sobrecarregar a memória.")